In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
from tensorflow import keras
from PIL import Image

model = keras.models.load_model("/content/drive/MyDrive/DS팀 데이터 저장소/oral_cnn.keras")

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

SAFE_TH = 0.30   # 30% 미만 = 안전
RISK_TH = 0.70   # 70% 이상 = 위험

def preprocess_image(path, size=(224, 224)):
    img = Image.open(path).convert("RGB").resize(size)
    x = np.array(img).astype(np.float32) / 255.0
    x = (x - MEAN) / STD
    x = np.expand_dims(x, axis=0)  # (1, H, W, C)
    return x

def risk_level(p: float) -> str:
    """
    기준:
    - p < 0.30         -> 안전
    - 0.30 <= p < 0.70 -> 주의
    - p >= 0.70        -> 위험
    """
    if p < SAFE_TH:
        return "안전"
    elif p < RISK_TH:
        return "주의"
    else:
        return "위험"

def predict_one(path: str):
    x = preprocess_image(path)
    p = float(model.predict(x, verbose=0).reshape(-1)[0])  # sigmoid prob (0~1)
    level = risk_level(p)
    return {
        "prob_cancer": p,
        "prob_percent": round(p * 100, 2),
        "level": level,
        "criteria": ">=70% 위험, 30~70% 주의, <30% 안전"
    }